To get a more realistic model of general TB dynamics, we need to account for the delayed onset of infectiousness upon infection, or the latent period. Ie., we need one or more compartments between "susceptible" and "infectious". 

In [ ]:
import pandas as pd
pd.options.plotting.backend = "plotly"
from summer2 import CompartmentalModel
from summer2.parameters import Parameter

In [ ]:
# Agnostic SEIR model
def build_seir_model(
        config: dict,
) -> CompartmentalModel:
    
    """ 
    Args: 
        config: dict with configurations other than parameters
    Returns:
        model: the summer model object
    """
    compartments = (
        "susceptible",
        "exposed",
        "infectious",
        "recovered",
    )
    analysis_times = (config["start_time"], config["end_time"])

    model = CompartmentalModel(
        times = analysis_times,
        compartments = compartments,
        infectious_compartments = ("infectious",),
    )

    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"], 
        },
    )

    # Parameters/flows
    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="exposed",
    )
    model.add_transition_flow(
        name="progression",
        fractional_rate=Parameter("progression"),
        source="exposed",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    return model

In [ ]:
# Configurations
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau
    "seed": 144.0, # Number of cases diagnosed in 2004
    "start_time": 2004,
    "end_time": 2020,
}

# Create instance of SEIR model with these configs
seir_model = build_seir_model(model_config)


In [8]:
# Flow parameters
parameters = {
    "recovery": 0.15, # -Ish for untreated TB per year
    "contact_rate": 10, # Effective contacts per person per year

    # Background mortality in 2004 was 1/45.5 = 0.0220. Let's say lifetime risk of progression is 10% as previous literature states, 
    # then progression rate should be k = 0.10(k + 0.0220) using the competing risks formula: risk = k/(k + mu), 
    # where k is the progression rate and mu is the background mortality rate
    "progression": 0.00244 
}

# Run the SEIR model with these specified parameters
seir_model.run(parameters=parameters)
compartment_values = seir_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)


Not surprisingly, this model also does not capture realistic TB dynamics. The average dwell time in the exposed compartment is 1/k, and with a progression rate of 0.00244, this is 410 years. The result is as the figure shows above; the population gradually moves from the susceptible to the exposed compartment, but during this short time span, only 279/48928 or around 0.5% of the total population ends up in the recovered compartment - and 144 were infectious (already passed the slow exposed compartment) to begin with. 

Newer studies suggest that the risk of infection and the average rate of progression to disease may be higher than we previously thought, and that many cases previously thought to arise from reactivation actually arise from reinfection. 

Let's say that the average dwell time in the exposed compartment is 1 year, then the rate of progression is exactly 1 (we ignore the mortality rate for now).

In [9]:
# Flow parameters
parameters = {
    "recovery": 0.15, # -Ish for untreated TB per year
    "contact_rate": 10, # Effective contacts per person per year

    # Latent period is 1 year on average
    "progression": 1.0
}

# Run the SEIR model with these specified parameters
seir_model.run(parameters=parameters)
compartment_values = seir_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

Now the exposed compartment rather acts as a short transit zone, and we get a TB epidemic that spikes within the first few years.

Let's continue with this basic model just for fun, even though it might not reflect the true underlying TB epidemiology. 